In [184]:
# =========================
# version_9
# N-HiTS model
# =========================

import os, re, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from numpy.lib.stride_tricks import sliding_window_view
from korean_lunar_calendar import KoreanLunarCalendar
from copy import deepcopy

plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False

# -------------------------
# 하이퍼파라미터/전역
# -------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 96, 50
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

PATIENCE = 10
MIN_SEQUENCE_COUNT = 10

# 업장 단종 (옵션)
DISCONTINUED = {
    '담하_꼬막_비빔밥': '2024-04-01',
    '담하_들깨_양지탕': '2024-04-01',
}
# -------------------------
# 추천 슬림 피처 세트
# -------------------------
FEATURES = [
    'clipped_SQ',        # scaled
    'rolling_mean_7',    # scaled
    'delta_scaled',      # scaled
    'holiday_prox',
    'is_holiday',
    'lag_7', 'lag_14','lag_28',
    'rel_level_7',
    'vol_14',
    'momentum_7',
    'ewm_mean_7',
    'doy_sin1', 'doy_cos1', 'doy_sin2', 'doy_cos2','doy_sin3', 'doy_cos3',
]
FEAT = {name: i for i, name in enumerate(FEATURES)}
for k in ['rolling_mean_7', 'holiday_prox', 'momentum_7', 'vol_14', 'ewm_mean_7']:
    assert k in FEAT, f"Missing required feature: {k}"
# ==== Fast holiday cache & proximity (drop-in) ====
HOLIDAYS_INT = None  # 전역 캐시: 'datetime64[D]'를 정수(일수)로
# -------------------------
# 캘린더(양/음력) 준비
# -------------------------
def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass
    return dates

years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1), (3, 1), (5, 5), (6, 6), (8, 15), (10, 3), (10, 9), (12, 25)
]

def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['is_solar_holiday'] = df['영업일자'].apply(lambda x: (x.month, x.day) in solar_md_list)
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    return df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])

# month_idx/season 만 추가 (Fourier 제거)
def add_month_idx_features(df: pd.DataFrame, date_col: str = '영업일자') -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    m = out[date_col].dt.month
    doy = out[date_col].dt.dayofyear.astype(np.int16)    # 1..365 (윤년은 무시해도 충분)
    # (B) 연간 주기: 365일 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        out[f'doy_sin{k}'] = np.sin(2*np.pi*k*doy/365).astype('float32')
        out[f'doy_cos{k}'] = np.cos(2*np.pi*k*doy/365).astype('float32')
    out['month_idx'] = (m - 1).astype(int)  # 0~11
    out['season'] = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)
    out['weekday'] = out[date_col].dt.dayofweek.astype(int)
    return out
# -------------------------
# Preprocessing
# -------------------------
def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환


def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용


def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)
# -------------------------
# store 학습 분리
# -------------------------
def _menu_only_from_key(key: str) -> str:
    """'영업장명_메뉴명' → '메뉴명'"""
    s = str(key)
    return s.split('_', 1)[1] if '_' in s else s

def make_store_context(df_store: pd.DataFrame) -> pd.DataFrame:
    """업장 전체 총매출 기반 컨텍스트 (store_rm7) 생성"""
    g = df_store.copy()
    g['영업일자'] = pd.to_datetime(g['영업일자'])
    agg = (g.groupby('영업일자', as_index=False)['매출수량'].sum()
             .sort_values('영업일자'))
    agg['store_rm7'] = agg['매출수량'].rolling(7, min_periods=1).mean().astype('float32')
    return agg[['영업일자','store_rm7']]

def _store_from_key(key: str) -> str:
    """'영업장명_메뉴명' → '영업장명'"""
    return str(key).split('_', 1)[0]
    
import math
def build_batches_inv_sqrt(item_ids: torch.Tensor, batch_size: int, seed: int | None = None):
    """
    각 미니배치가 '단일 메뉴'로만 구성되도록 인덱스를 묶어 반환.
    메뉴 선택은 남은 샘플이 있는 메뉴들 중에서 가중치 w_i = 1/sqrt(N_i) 로 랜덤 선택.
    모든 샘플을 정확히 한 번씩 소비합니다.
    Returns: list[(menu_id:int, batch_idx: LongTensor)]
    """
    rnd = random.Random(seed) if seed is not None else random

    # 1) 메뉴별 버킷
    ids = item_ids.detach().cpu().tolist()
    buckets = {}  # menu_id -> [sample_indices...]
    for i, it in enumerate(ids):
        buckets.setdefault(int(it), []).append(i)

    menu_ids = list(buckets.keys())

    # 2) 버킷 내부 셔플
    for m in menu_ids:
        rnd.shuffle(buckets[m])

    # 3) 1/sqrt(N) 가중
    weights = {m: 1.0 / max(math.sqrt(len(buckets[m])), 1.0) for m in menu_ids}

    # 4) 남은 샘플이 있는 메뉴들만 대상으로 가중 랜덤 선택하며 배치 생성
    ptr = {m: 0 for m in menu_ids}
    active = [m for m in menu_ids if len(buckets[m]) > 0]
    batches = []

    while active:
        # 활성 메뉴들에 대해 누적합으로 가중 랜덤 선택
        ws = [weights[m] for m in active]
        total = sum(ws)
        r = rnd.random() * total
        cum = 0.0
        chosen = active[0]
        for m, wm in zip(active, ws):
            cum += wm
            if r <= cum:
                chosen = m
                break

        # 선택된 메뉴에서 batch_size 만큼 꺼내기
        start = ptr[chosen]
        end = min(start + batch_size, len(buckets[chosen]))
        idxs = buckets[chosen][start:end]
        ptr[chosen] = end
        if ptr[chosen] >= len(buckets[chosen]):
            active.remove(chosen)

        batches.append((chosen, torch.tensor(idxs, dtype=torch.long, device=item_ids.device)))

    return batches
# -------------------------
# 유틸/전처리
# -------------------------
def ensure_time_major(x: torch.Tensor, lookback: int, n_features: int) -> torch.Tensor:
    if x.dim() != 3:
        raise ValueError(f"Expected 3D tensor, got {x.dim()}D: {tuple(x.shape)}")
    B, A, B2 = x.shape
    if A == lookback and B2 == n_features:
        return x
    if A == n_features and B2 == lookback:
        return x.permute(0, 2, 1).contiguous()
    raise ValueError(f"Unexpected shape {tuple(x.shape)} (T={lookback}, F={n_features})")

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='min',
                 restore_best_weights=True, min_epochs=0, relative=True, smooth_beta=0.0):
        self.patience = patience; self.min_delta = min_delta; self.mode = mode
        self.restore_best_weights = restore_best_weights
        self.min_epochs = min_epochs; self.relative = relative; self.smooth_beta = smooth_beta
        self.best = None; self.best_state = None; self.wait = 0; self.stop = False; self.ema = None

    def _improved(self, current):
        if self.best is None: return True
        if self.mode == 'min':
            return current < (self.best * (1.0 - self.min_delta) if self.relative else (self.best - self.min_delta))
        else:
            return current > (self.best * (1.0 + self.min_delta) if self.relative else (self.best + self.min_delta))

    def step(self, current, model, epoch_idx: int):
        monitor_val = current if self.smooth_beta == 0 else (self.smooth_beta*(self.ema or current) + (1-self.smooth_beta)*current)
        self.ema = monitor_val
        if self.best is None or self._improved(monitor_val):
            self.best = monitor_val; self.best_state = deepcopy(model.state_dict()); self.wait = 0; return False
        self.wait += 1
        if epoch_idx+1 < self.min_epochs: return False
        if self.wait >= self.patience:
            self.stop = True
            if self.restore_best_weights and self.best_state is not None:
                model.load_state_dict(self.best_state)
            return True
        return False

def clip_df(series, q=0.9995):
    s = pd.Series(series, copy=False).astype('float32')
    ub = float(np.nanquantile(s, q))
    return s.clip(upper=ub)

def add_holiday_proximity(
    df: pd.DataFrame, date_col='영업일자', holiday_col='is_holiday', out_col='holiday_prox', K=10
) -> pd.DataFrame:
    df = df.copy()
    if date_col not in df or holiday_col not in df: return df
    orig_index = df.index
    tmp = df[[date_col, holiday_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col]); tmp = tmp.sort_values(date_col)
    mask = tmp[holiday_col].astype(bool); s_h = tmp[date_col].where(mask)
    prev_h, next_h = s_h.ffill(), s_h.bfill()
    dist_prev = (tmp[date_col] - prev_h).dt.days.astype('float32').fillna(K+1)
    dist_next = (next_h - tmp[date_col]).dt.days.astype('float32').fillna(K+1)
    dist_h = np.minimum(dist_prev, dist_next).clip(0, K).astype('float32')
    out = ((K - dist_h) / K).astype('float32')
    out_df = pd.DataFrame({out_col: out}, index=tmp.index).reindex(orig_index)
    df[out_col] = out_df[out_col].values.astype('float32')
    return df

def add_ts_stats(
    df: pd.DataFrame, target_col="clipped_SQ", date_col="영업일자",
    lags=(7,14,28), roll_windows=(7,14), ewm_spans=(7,), eps=1e-3
) -> pd.DataFrame:
    df = df.sort_values(date_col).copy()
    x = df[target_col].astype('float32')

    # lags
    for k in lags:
        df[f"lag_{k}"] = x.shift(k).astype('float32')

    # rolling mean/std (현재 포함; 엄격히 하려면 x.shift(1)로 교체)
    for w in roll_windows:
        df[f"roll_mean_{w}"] = x.rolling(w, min_periods=1).mean().astype('float32')
        df[f"roll_std_{w}"]  = x.rolling(w, min_periods=1).std().fillna(0).astype('float32')

    # EWM
    for s in ewm_spans:
        df[f"ewm_mean_{s}"] = x.ewm(span=s, adjust=False).mean().astype('float32')

    # momentum/rel_level/vol
    for k in (7,):
        df[f"momentum_{k}"] = ((x - x.shift(k)) / (np.abs(x.shift(k)) + eps)).astype('float32')
    for w in (7,14):
        m = df[f"roll_mean_{w}"]; s = df[f"roll_std_{w}"]
        df[f"rel_level_{w}"] = (x / (m + eps)).astype('float32')
        df[f"vol_{w}"]       = (s / (m + eps)).astype('float32')
    return df

def inverse_clipped_from_scaler(scaler_xy: MinMaxScaler, scaled_vals: np.ndarray) -> np.ndarray:
    dummy = np.zeros((len(scaled_vals), 2), dtype=np.float32)
    dummy[:, 0] = scaled_vals
    return scaler_xy.inverse_transform(dummy)[:, 0]

def estimate_delta_from_y(y_real, clip_min: float = 10.0, clip_max: float = 80.0) -> float:
    y = np.asarray(y_real, dtype=float); y = y[np.isfinite(y)]
    if y.size == 0: return float((clip_min + clip_max) / 2.0)
    med = np.median(y); mad = np.median(np.abs(y - med)); mad = max(mad, 1e-6)
    delta = 1.35 * mad; return float(np.clip(delta, clip_min, clip_max))

# -------------------------
# Loss
# -------------------------
def smape_real(yhat, y, eps=1.0, ignore_zero_target=True):
    num = (yhat - y).abs()
    den = (yhat.abs() + y.abs()).clamp_min(eps)
    sm = 2.0 * num / den
    if ignore_zero_target:
        mask = (y > 0).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        return ((sm * mask).sum(dim=1) / denom).mean()
    return sm.mean()

def weighted_huber_smape(
    yhat, y, w=None, *, delta=0.05, eps=1e-3, alpha=0.5
):
    hub = F.huber_loss(yhat, y, delta=delta, reduction='none')
    den = yhat.abs() + y.abs()
    if isinstance(eps, float) or isinstance(eps, int):
        den = torch.clamp(den, min=float(eps))
    else:
        den = torch.maximum(den, eps)
    smp = 2.0 * (yhat - y).abs() / den
    loss = alpha * hub + (1 - alpha) * smp
    if w is not None: loss = loss * w
    return loss.mean()

def smape_loss(pred, target, eps=1e-3, reduction='mean', ignore_zero_target=True):
    num = (pred - target).abs()
    den = (pred.abs() + target.abs()).clamp_min(eps)
    sm = 2.0 * num / den
    if ignore_zero_target:
        mask = (target != 0).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        row_mean = (sm * mask).sum(dim=1) / denom
        return row_mean.mean()
    return sm.mean()

def visualize_loss(train_losses, val_losses, store_menu, save=False, out_dir="./loss_plots",
                   show=False, verbose=False, val_smape=None):
    import numpy as np
    plt.figure(figsize=(6,4)); ax = plt.gca()
    def to_float_array(xs):
        if xs is None: return np.array([], dtype=float)
        try: return np.asarray([float(x) for x in xs], dtype=float)
        except Exception: return np.array(xs, dtype=float)
    tr = to_float_array(train_losses); va = to_float_array(val_losses)
    tr_mask = np.isfinite(tr); va_mask = np.isfinite(va)
    drew = False
    if tr.size > 0 and tr_mask.any():
        ax.plot(np.arange(1, tr.size+1)[tr_mask], tr[tr_mask], marker='o', lw=1.5, label='Train Loss'); drew=True
    if va.size > 0 and va_mask.any():
        ax.plot(np.arange(1, va.size+1)[va_mask], va[va_mask], marker='o', lw=1.5, label='Validation Loss'); drew=True
    title = f"[{store_menu}] Train vs Validation Loss"; ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    if val_smape is not None and len(val_smape) > 0:
        ax2 = ax.twinx(); sm = to_float_array(val_smape); msk = np.isfinite(sm)
        if sm.size > 0 and msk.any():
            ax2.plot(np.arange(1, sm.size+1)[msk], sm[msk], ls='--', marker='x', lw=1.2, label='Val sMAPE (scaled)')
        ax2.set_ylabel("sMAPE"); ax2.grid(False)
        l1, t1 = ax.get_legend_handles_labels(); l2, t2 = ax2.get_legend_handles_labels()
        ax2.legend(l1+l2, t1+t2, loc='upper right')
    else:
        if drew: ax.legend(loc='upper right')
    if drew:
        ax.grid(True, alpha=0.4)
        ymin = min(np.min(tr[tr_mask]) if tr_mask.any() else np.inf,
                   np.min(va[va_mask]) if va_mask.any() else np.inf)
        ymax = max(np.max(tr[tr_mask]) if tr_mask.any() else -np.inf,
                   np.max(va[va_mask]) if va_mask.any() else -np.inf)
        if np.isfinite(ymin) and np.isfinite(ymax) and ymin != ymax:
            pad = 0.05 * (ymax - ymin); ax.set_ylim(ymin - pad, ymax + pad)
    else:
        ax.grid(True, alpha=0.4); ax.text(0.5,0.5,"No points to plot", ha='center', va='center', transform=ax.transAxes)
    safe_name = re.sub(r'[^\w\-_.]', '_', (store_menu if isinstance(store_menu,str) else "_".join(map(str,store_menu))))
    if save:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"{safe_name}.png")
        plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight')
    if show and not save:
        plt.tight_layout(); plt.show()
    plt.close()
def smape_per_key(y_true, y_pred, keys):
    num = np.abs(y_pred - y_true)
    den = np.clip(np.abs(y_pred) + np.abs(y_true), 1e-3, None)
    sm = 200.0 * num / den
    df = pd.DataFrame({'key':keys, 'smape':sm})
    return df.groupby('key')['smape'].mean().sort_values()
# -------------------------
# 빌드 피처 (슬림)
# -------------------------
def build_features(
    df: pd.DataFrame,
    scaler_y: MinMaxScaler,
    scaler_rm: MinMaxScaler,
    scaler_delta: MinMaxScaler,
    fit: bool = False,
    date_col: str = '영업일자'
) -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out = add_month_idx_features(out, date_col)
    out = generate_combined_holiday_list(out, solar_md_holidays, lunar_solar_dates)
    out = add_holiday_proximity(out, date_col, 'is_holiday', 'holiday_prox', K=10)

    # target clip & deltas
    out['clipped_SQ']     = clip_df(out['매출수량']) if '매출수량' in out.columns else out['clipped_SQ']
    out['delta_scaled']          = out['clipped_SQ'].diff().fillna(0)
    out['rolling_mean_7'] = out['clipped_SQ'].rolling(window=7, min_periods=1).mean()

    # ts stats (과거만)
    out = add_ts_stats(out, target_col="clipped_SQ", date_col=date_col,
                       lags=(7,14,28), roll_windows=(7,14), ewm_spans=(7,))
    # scaling
    if fit:
        scaler_y.fit(out[['clipped_SQ']].to_numpy())
        scaler_rm.fit(out[['rolling_mean_7']].to_numpy())
        scaler_delta.fit(out[['delta_scaled']].to_numpy())

    out['clipped_SQ']     = scaler_y.transform(out[['clipped_SQ']].to_numpy())[:,0]
    out['rolling_mean_7'] = scaler_y.transform(out[['rolling_mean_7']].to_numpy())[:,0]   # ← 통일 권장
    out['delta_scaled']   = scaler_delta.transform(out[['delta_scaled']].to_numpy())[:,0]

    for k in (7,14,28):
        out[f'lag_{k}']   = scaler_y.transform(out[[f'lag_{k}']].to_numpy())[:,0]         # ← 본인 컬럼!
    out['ewm_mean_7']     = scaler_y.transform(out[['ewm_mean_7']].to_numpy())[:,0]       # ← 본인 컬럼!


    # 필요 컬럼만 유지(없으면 0으로 채움)
    cols_need = set(FEATURES + ['weekday','season','month_idx', date_col, '매출수량'])
    for c in (set(out.columns) - cols_need):
        pass  # 나머지 컬럼은 있어도 무시됨
    # 결측/타입 정리
    use_cols = [c for c in FEATURES if c in out.columns]
    out[use_cols] = (out[use_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0).astype('float32'))
    out['is_holiday'] = out['is_holiday'].astype('float32')
    return out

# -------------------------
# 미래 달력 유틸
# -------------------------
def build_holidays_int(years):
    import numpy as np
    dates = []
    for y in years:
        for (m, d) in solar_md_holidays:
            try:
                dates.append(pd.Timestamp(year=y, month=m, day=d))
            except Exception:
                pass
    lunar_list = pd.to_datetime(lunar_solar_dates, errors='coerce')

    # 합치고 중복 제거
    holidays = pd.to_datetime(
        pd.Index(dates).append(pd.Index(lunar_list)),
        errors='coerce'
    ).dropna().unique()

    # 타임존 제거(있다면) 후 numpy로 변환
    holidays = pd.DatetimeIndex(holidays).tz_localize(None)

    # 방법 A(가장 호환성 좋음): epoch ns → days로 변환
    ns = holidays.to_numpy(dtype='datetime64[ns]')
    days = (ns - np.datetime64('1970-01-01', 'ns')) // np.timedelta64(1, 'D')
    return days.astype('int64')


def holiday_prox_searchsorted(dates_pd_index, holidays_int, K=10):
    """
    dates_pd_index: DatetimeIndex or array-like of pandas Timestamps
    holidays_int  : np.ndarray[int64], sorted (days since epoch)
    """
    d_int = pd.to_datetime(dates_pd_index).values.astype('datetime64[D]').astype('int64')
    idx = np.searchsorted(holidays_int, d_int, side='left')  # 다음 휴일 위치
    M = holidays_int.size

    # prev/next 인덱스 클립
    prev_idx = np.clip(idx - 1, 0, max(M - 1, 0))
    next_idx = np.clip(idx,       0, max(M - 1, 0))

    # 거리 계산(일수)
    dist_prev = d_int - holidays_int[prev_idx]
    dist_next = holidays_int[next_idx] - d_int

    # 경계 처리: 유효하지 않으면 K+1로
    dist_prev = np.where(idx > 0, dist_prev, K + 1)
    dist_next = np.where(idx < M, dist_next, K + 1)

    mind = np.minimum(dist_prev, dist_next)
    mind = np.clip(mind, 0, K)
    prox = (K - mind) / float(K)
    return prox.astype(np.float32)
def make_future_calendar_fast(last_dates_np, horizon, holidays_int, K=10):
    """
    last_dates_np: np.ndarray(datetime64[ns] 또는 pandas Timestamps), shape (S,)
    return: (f_wd, f_mm, f_px) each (S, H)
    """
    S = len(last_dates_np)
    # (S,H) 미래 날짜
    base = pd.to_datetime(last_dates_np).values.astype('datetime64[D]')
    fut = base[:, None] + np.arange(1, horizon + 1, dtype='timedelta64[D]')[None, :]
    fut_pd = pd.to_datetime(fut.reshape(-1))  # (S*H,)

    wd = fut_pd.dayofweek.values.reshape(S, horizon).astype(np.int64)
    mm = (fut_pd.month.values.reshape(S, horizon) - 1).astype(np.int64)  # 0~11

    prox_flat = holiday_prox_searchsorted(fut_pd, holidays_int, K=K)      # (S*H,)
    px = prox_flat.reshape(S, horizon)
    return wd, mm, px
def build_holiday_index(solar_md_holidays, lunar_solar_dates, years):
    solar_list = []
    for y in years:
        for (m,d) in solar_md_holidays:
            try: solar_list.append(pd.Timestamp(year=y, month=m, day=d))
            except: pass
    lunar_list = pd.to_datetime(lunar_solar_dates, errors='coerce')
    holi = pd.to_datetime(pd.Index(solar_list).append(pd.Index(lunar_list)).unique()).sort_values()
    return holi

def make_future_calendar(last_dates: np.ndarray, horizon: int, holiday_index: pd.DatetimeIndex, K: int = 10):
    S = len(last_dates)
    base = last_dates.astype('datetime64[D]')[:, None]
    offs = np.arange(1, horizon+1, dtype='timedelta64[D]')[None, :]
    fut = (base + offs).astype('datetime64[D]')              # (S,H)
    fut_pd = pd.to_datetime(fut.reshape(-1))                 # (S*H,)
    wd = fut_pd.dayofweek.values.reshape(S,horizon).astype(np.int64)
    mm = (fut_pd.month.values.reshape(S,horizon) - 1).astype(np.int64)  # 0~11
    if len(holiday_index)==0:
        prox = np.zeros((S,horizon), dtype=np.float32)
    else:
        fut_d = fut_pd.values.astype('datetime64[D]')[:, None]
        hol_d = holiday_index.values.astype('datetime64[D]')
        dist = np.abs(fut_d - hol_d).astype('timedelta64[D]').astype(np.int32)
        mind = dist.min(axis=1); mind = np.clip(mind, 0, K)
        prox = ((K - mind) / float(K)).astype(np.float32).reshape(S,horizon)
    return wd, mm, prox
# -------------------------
# 유틸리티
# -------------------------
def softplus_inv(x: float) -> float:
    # numerically stable inverse of softplus
    return math.log(math.expm1(x))

def make_mixed_base(Xb, H):
    Xb = ensure_time_major(Xb, LOOKBACK, len(FEATURES))
    rm7  = Xb[:, -1, FEAT['rolling_mean_7']].unsqueeze(1).expand(-1, H)
    ewm7 = Xb[:, -1, FEAT['ewm_mean_7']].unsqueeze(1).expand(-1, H)
    s7   = Xb[:, -H:, FEAT['clipped_SQ']]
    mom  = Xb[:, -1, FEAT['momentum_7']]
    prox = Xb[:, -1, FEAT['holiday_prox']]
    vol  = Xb[:, -1, FEAT['vol_14']]

    g_vol = (vol - 0.2).clamp(0, 0.8) / 0.8
    g_mom = mom.clamp(0, 0.4) / 0.4
    g_szn = prox.clamp(0, 1)

    w_s7_raw = 0.6 + 0.3*g_vol + 0.1*g_szn
    w_ew_raw = 0.2 + 0.5*g_mom + 0.2*(1 - g_szn)
    w_rm_raw = 0.2
    w_sum = w_s7_raw + w_ew_raw + w_rm_raw

    w_s7 = (w_s7_raw / w_sum).unsqueeze(1).expand_as(s7)
    w_ew = (w_ew_raw / w_sum).unsqueeze(1).expand_as(ewm7)
    w_rm = (w_rm_raw / w_sum).unsqueeze(1).expand_as(rm7)
    return w_rm*rm7 + w_ew*ewm7 + w_s7*s7

# -------------------------
# 모델
# -------------------------
class MRBlock(nn.Module):
    def __init__(self, in_dim, horizon, pool_k: int, mlp_dim=128, mlp_layers=2):
        super().__init__()
        self.pool_k = pool_k
        layers = [nn.Linear(in_dim, mlp_dim), nn.ReLU()]
        for _ in range(mlp_layers-1):
            layers += [nn.Linear(mlp_dim, mlp_dim), nn.ReLU()]
        self.mlp = nn.Sequential(*layers)
        self.norm = nn.LayerNorm(mlp_dim)
        self.drop = nn.Dropout(p=0.1)
        self.proj = nn.Linear(mlp_dim * 2, horizon)

    def forward(self, x):                # x: (B,T,F)
        x_ch = x.transpose(1, 2)         # (B,F,T)
        x_pool = F.avg_pool1d(x_ch, kernel_size=self.pool_k, stride=self.pool_k, ceil_mode=True) if self.pool_k>1 else x_ch
        x_pool = x_pool.transpose(1, 2)  # (B,T',F)
        z = self.mlp(x_pool)             # (B,T',D)
        z = self.norm(z); z = self.drop(z)
        z_mean, z_max = z.mean(dim=1), z.amax(dim=1)
        z_cat = torch.cat([z_mean, z_max], dim=-1)          # (B,2D)
        return self.proj(z_cat)                              # (B,H)

class NHiTSWithEmbeddingMR(nn.Module):
    """
    두-헤드 구조:
      - prob_head: "발생 확률" (요일/휴일/월 패턴을 주로 학습)
      - mag_head : "크기 residual" (지난주 베이스 주변만 미세 보정)
    """
    def __init__(self, lookback, input_dim, horizon,
                 pools=(28,7,1),
                 weekday_vocab=7, weekday_emb_dim=2,
                 season_vocab=4,  season_emb_dim=2,
                 month_vocab=12,  month_emb_dim=3,
                 emb_dropout=0.10, use_sigmoid_output=False,
                 mr_mlp_dim=96):
        super().__init__()
        self.input_dim = input_dim
        self.lookback, self.horizon = lookback, horizon
        self.use_residual_base = True
        self.use_sigmoid_output = use_sigmoid_output  # (미사용 권장)

        # softplus 역변환 이미 유틸에 있음: softplus_inv
        self.r_scale_param = nn.Parameter(torch.tensor(softplus_inv(0.10), dtype=torch.float32))

        # ----- 달력 임베딩 -----
        self.weekday_emb = nn.Embedding(weekday_vocab, weekday_emb_dim)
        self.season_emb  = nn.Embedding(season_vocab,  season_emb_dim)
        self.month_emb   = nn.Embedding(month_vocab,   month_emb_dim)
        cat_dim = weekday_emb_dim + season_emb_dim + month_emb_dim

        self.post_emb_norm = nn.LayerNorm(input_dim + cat_dim)
        self.post_emb_drop = nn.Dropout(emb_dropout)

        # ----- MR Blocks (그대로) -----
        self.blocks = nn.ModuleList([
            MRBlock(in_dim=input_dim + cat_dim, horizon=horizon, pool_k=p, mlp_dim=mr_mlp_dim, mlp_layers=2)
            for p in pools
        ])

        # ----- 미래 H별 임베딩 -----
        self.h_emb = nn.Embedding(horizon, 8)
        self.wd_future_emb = nn.Embedding(7, 8)
        self.mm_future_emb = nn.Embedding(12, 4)
        self.prox_lin = nn.Linear(1, 2)

        # ----- 두-헤드 -----
        aux_dim = 8 + 8 + 4 + 2        # he(8) + wd(8) + mm(4) + prox(2) = 22
        in_dim  = 1 + aux_dim          # rep(1) + aux
        self.prob_head = nn.Sequential(nn.Linear(in_dim, 64), nn.GELU(), nn.Linear(64, 1))
        self.mag_head  = nn.Sequential(nn.Linear(in_dim,128), nn.GELU(), nn.Linear(128,1))

        # 게이트 분위수(학습에서 덮어씀)
        self.register_buffer("gate_p10", torch.tensor(0.10, dtype=torch.float32))
        self.register_buffer("gate_p90", torch.tensor(0.90, dtype=torch.float32))
        self.gate_floor = 0.15

        self.dow_bias = nn.Parameter(torch.zeros(7))


    def _gate_from_level(self, level: torch.Tensor) -> torch.Tensor:
        # level: (B,)
        p10, p90 = self.gate_p10, self.gate_p90
        k = 8.0 / (p90 - p10 + 1e-6)
        gate = torch.sigmoid((level - p10) * k).unsqueeze(1)    # (B,1)
        return torch.clamp(gate, min=self.gate_floor)
    
    def forward(self, x_num, x_weekday, x_season, x_month,
                future_weekday=None, future_month=None, future_prox=None,
                base_last=None, return_components: bool = False):
        
        # 입력/임베딩 결합
        x_num = ensure_time_major(x_num, self.lookback, self.input_dim)
        x_num = x_num.contiguous(); x_weekday = x_weekday.contiguous()
        x_season = x_season.contiguous(); x_month = x_month.contiguous()
        B = x_num.size(0)

        w = self.weekday_emb(x_weekday); s = self.season_emb(x_season); m = self.month_emb(x_month)
        x = torch.cat([x_num, w, s, m], dim=-1)
        x = self.post_emb_norm(x); x = self.post_emb_drop(x)

        reps = [b(x) for b in self.blocks]            # list of (B,H)
        rep  = torch.stack(reps, dim=0).mean(dim=0)   # (B,H)
        B, H = rep.size(); device = rep.device
            
        if future_weekday is None:
            future_weekday = (x_weekday[:, -1].unsqueeze(1) + torch.arange(1, H+1, device=device)) % 7
        if future_month is None:
            future_month = x_month[:, -1].unsqueeze(1).expand(B, H)
        if future_prox is None:
            future_prox = torch.zeros(B, H, device=device)
        
        he = self.h_emb(torch.arange(H, device=device).unsqueeze(0).expand(B, -1))
        we = self.wd_future_emb(future_weekday)
        me = self.mm_future_emb(future_month)
        pe = self.prox_lin(future_prox.unsqueeze(-1))
        aux = torch.cat([he, we, me, pe], dim=-1)     # (B,H,22)

        rep_exp = rep.unsqueeze(-1)                   # (B,H,1)
        feat = torch.cat([rep_exp, aux], dim=-1)      # (B,H,1+22)

        # level & gate
        level_now  = x_num[:, -1, FEAT['rolling_mean_7']]
        level_week = x_num[:, -7:, FEAT['clipped_SQ']].mean(dim=1)
        level = torch.maximum(level_now, level_week)  # (B,)
        gate  = self._gate_from_level(level)          # (B,1)

        # base (게이트 곱하지 않음)
        base = x_num[:, -1, FEAT['rolling_mean_7']] if base_last is None else base_last
        if base.dim() == 1:
            base = base.unsqueeze(1).expand(B, H)

        bias = torch.tanh(self.dow_bias)[future_weekday]

        # 두-헤드
        prob_logit = self.prob_head(feat).squeeze(-1)    # (B,H)
        resid_raw  = self.mag_head(feat).squeeze(-1)     # (B,H)

        r_scale = F.softplus(self.r_scale_param)
        resid   = gate * torch.tanh(resid_raw)           # 베이스 주변만
        mag  = torch.relu(base + r_scale * resid)
        mag  = mag * (1.0 + 0.30 * bias)     # ✅ 출력단에서 요일 보정 (±30% 권장)

        y_hat = mag          # 기대값(확률 * 크기)

        if return_components:
            return y_hat, prob_logit, resid_raw, base
        return y_hat
# -------------------------
# Train
# -------------------------
def train_nhits_itemwise(train_df: pd.DataFrame,
                         use_validation: bool = True,
                         lr: float = 8e-4,
                         weight_decay: float = 1e-5,
                         max_grad_norm: float = 1.0,
                         emb_dropout: float = 0.10,
                         mr_mlp_dim: int = 96,
                         grad_accum_steps: int = 1,
                         plot_dir: str = "./loss_plots_item",
                         lambda_cls: float = 0.05,   # ✅ BCE 비중 축소(0~0.05 권장)
                         lambda_mag: float = 0.45,   # ✅ 양성일 Huber/MSE
                         lambda_ev:  float = 0.50,   # ✅ real sMAPE(0-day 제외)
                         store_gain: dict | None = None  # 예: {'담하':1.3, '미라시아':1.3}
                         ):
    if store_gain is None:
        store_gain = {'담하': 1.3, '미라시아': 1.3}

    trained = {}

    df = train_df.copy()
    if '영업장명' not in df.columns:
        df['영업장명'] = df['영업장명_메뉴명'].apply(_store_from_key)
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    # 휴일 인덱스 준비(빠른 캘린더)
    global HOLIDAYS_INT
    if HOLIDAYS_INT is None:
        years = df['영업일자'].dt.year
        HOLIDAYS_INT = build_holidays_int(range(int(years.min()) - 1, int(years.max()) + 2))

    menus_groups = list(df.groupby('영업장명_메뉴명', sort=False))
    with tqdm(total=len(menus_groups), desc='Training (item-wise)', dynamic_ncols=True) as pbar:
        for menu_key, g_menu in menus_groups:
            g_menu = g_menu.sort_values('영업일자')

            if len(g_menu) < LOOKBACK + PREDICT + MIN_SEQUENCE_COUNT:
                pbar.set_postfix_str(f"{_menu_only_from_key(menu_key)[:28]} (skipped)")
                pbar.update(1)
                continue

            cut = int(round(len(g_menu) * 0.8)) if use_validation else len(g_menu)
            cut = max(cut, LOOKBACK)
            cut -= cut % 7
            fit_part = g_menu.iloc[:cut].copy()

            # ✅ 스케일러 fit은 메뉴 학습 구간만
            scaler_y, scaler_rm, scaler_delta = MinMaxScaler(), MinMaxScaler(), MinMaxScaler()

            # ⚠️ build_features는 '원 단위→통계→스케일' 순서로 바꿔둔 버전 사용 권장
            fit_feat = build_features(fit_part, scaler_y, scaler_rm, scaler_delta, fit=True,  date_col='영업일자')
            feat    = build_features(g_menu,  scaler_y, scaler_rm, scaler_delta, fit=False, date_col='영업일자')

            # 게이트 분위수(레벨 분포) 산출
            lvl_now  = fit_feat['rolling_mean_7'].values.astype(np.float32)
            lvl_week = (
                pd.Series(fit_feat['clipped_SQ'])
                .rolling(7, min_periods=1).mean()
                .bfill().ffill()
                .to_numpy(dtype=np.float32, copy=False)
            )
            lvl = np.maximum(lvl_now, lvl_week)
            p10_fit = float(np.clip(np.quantile(lvl, 0.10), 1e-4, 0.5))
            p90_fit = float(np.clip(np.quantile(lvl, 0.90), p10_fit + 1e-4, 0.99))

            vals = feat[FEATURES].values.astype(np.float32)
            tgt  = feat['clipped_SQ'].values.astype(np.float32)
            wd   = feat['weekday'].values.astype(np.int64)
            ss   = feat['season'].values.astype(np.int64)
            mm   = feat['month_idx'].values.astype(np.int64)

            total_seq = len(feat) - LOOKBACK - PREDICT + 1
            if total_seq <= 0:
                pbar.set_postfix_str(f"{_menu_only_from_key(menu_key)[:28]} (no_window)")
                pbar.update(1)
                continue

            X_np  = sliding_window_view(vals, LOOKBACK, axis=0)[:total_seq]
            y_np  = sliding_window_view(tgt,  LOOKBACK+PREDICT, axis=0)[:total_seq, LOOKBACK:]
            wd_np = sliding_window_view(wd,   LOOKBACK, axis=0)[:total_seq]
            ss_np = sliding_window_view(ss,   LOOKBACK, axis=0)[:total_seq]
            mm_np = sliding_window_view(mm,   LOOKBACK, axis=0)[:total_seq]

            dates_np = feat['영업일자'].values.astype('datetime64[ns]')
            last_dates_all = pd.to_datetime(dates_np[LOOKBACK - 1 : LOOKBACK - 1 + total_seq]).values
            fwd_np, fmm_np, fpx_np = make_future_calendar_fast(last_dates_all, PREDICT, HOLIDAYS_INT, K=10)

            # Tensorize
            X   = torch.tensor(X_np,   dtype=torch.float32, device=DEVICE)
            y   = torch.tensor(y_np,   dtype=torch.float32, device=DEVICE)
            wdT = torch.tensor(wd_np,  dtype=torch.long,    device=DEVICE)
            ssT = torch.tensor(ss_np,  dtype=torch.long,    device=DEVICE)
            mmT = torch.tensor(mm_np,  dtype=torch.long,    device=DEVICE)
            fwd = torch.tensor(fwd_np, dtype=torch.long,    device=DEVICE)
            fmm = torch.tensor(fmm_np, dtype=torch.long,    device=DEVICE)
            fpx = torch.tensor(fpx_np, dtype=torch.float32, device=DEVICE)

            # Split
            split = int(len(X) * 0.8) if use_validation else len(X)
            split -= split % 7
            Xtr, Xval   = X[:split],   X[split:]
            ytr, yval   = y[:split],   y[split:]
            wdtr, wdval = wdT[:split], wdT[split:]
            sstr, ssval = ssT[:split], ssT[split:]
            mmtr, mmval = mmT[:split], mmT[split:]
            fwd_tr, fwd_val = fwd[:split], fwd[split:]
            fmm_tr, fmm_val = fmm[:split], fmm[split:]
            fpx_tr, fpx_val = fpx[:split], fpx[split:]

            # Model
            model = NHiTSWithEmbeddingMR(
                lookback=LOOKBACK, input_dim=len(FEATURES), horizon=PREDICT,
                pools=(28,7,1), emb_dropout=emb_dropout, use_sigmoid_output=False,
                mr_mlp_dim=mr_mlp_dim
            ).to(DEVICE)

            # ✅ 게이트 분위수/게이트 바닥/잔차 스케일 초기값 강화
            model.register_buffer("gate_p10", torch.tensor(p10_fit, dtype=torch.float32))
            model.register_buffer("gate_p90", torch.tensor(p90_fit, dtype=torch.float32))
            model.gate_floor = 0.35                          # 0.15 → 0.35
            with torch.no_grad():
                model.r_scale_param.copy_(torch.tensor(softplus_inv(0.8), device=DEVICE))  # 초기 진폭 ↑

            opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=4)
            early = EarlyStopping(patience=PATIENCE, min_epochs=8, restore_best_weights=True)

            # 실수 도메인 복원 상수
            y_min = torch.tensor(float(scaler_y.data_min_[0]), device=DEVICE)
            y_max = torch.tensor(float(scaler_y.data_max_[0]), device=DEVICE)
            scale = (y_max - y_min)

            # 스토어 가중
            store_name = _store_from_key(menu_key)
            g_store = float(store_gain.get(store_name, 1.0))

            train_losses, val_losses, val_smape_list = [], [], []
            for ep in range(EPOCHS):
                model.train()
                idx = torch.randperm(len(Xtr), device=DEVICE)
                sum_loss, n_obs, step_mod = 0.0, 0, 0
                opt.zero_grad(set_to_none=True)

                for i in range(0, len(Xtr), BATCH_SIZE):
                    b = idx[i:i+BATCH_SIZE]
                    Xb, yb   = Xtr[b], ytr[b]
                    wdb, ssb = wdtr[b], sstr[b]
                    mmb      = mmtr[b]
                    future_wd = fwd_tr[b]; future_mm = fmm_tr[b]; future_px = fpx_tr[b]

                    # 컴포넌트 출력
                    y_hat, prob_logit, resid_raw, base_mix = model(
                        Xb, wdb, ssb, mmb,
                        future_weekday=future_wd, future_month=future_mm, future_prox=future_px,
                        base_last=make_mixed_base(Xb, PREDICT),
                        return_components=True
                    )
                    ybin = (yb > 0).float()
                    mask_pos = (ybin > 0.5)

                    # ✅ magnitude(크기) 재계산 → 학습은 mag 중심
                    r_scale = F.softplus(model.r_scale_param)
                    pred_mag = torch.relu(base_mix + r_scale * torch.tanh(resid_raw))  # (B,H)

                    # ===== Loss 구성 =====
                    # (1) BCE 약화 (원하면 0으로 꺼도 됨)
                    pos_rate = ybin.mean().clamp(1e-3, 1 - 1e-3)
                    pos_weight = ((1 - pos_rate) / pos_rate).clamp(1.0, 4.0)
                    loss_cls = F.binary_cross_entropy_with_logits(
                        prob_logit, ybin, pos_weight=pos_weight
                    ).mean() * lambda_cls

                    # (2) Magnitude 회귀(양성일만, 스케일 도메인)
                    if mask_pos.any():
                        loss_mag = F.smooth_l1_loss(pred_mag[mask_pos], yb[mask_pos]) * lambda_mag
                    else:
                        loss_mag = torch.tensor(0.0, device=yb.device)

                    # (3) real sMAPE (0-day 제외, 실제 단위)
                    yhat_real = pred_mag * scale + y_min
                    yb_real   = yb       * scale + y_min
                    loss_ev   = smape_real(yhat_real, yb_real, eps=1.0, ignore_zero_target=True) * lambda_ev

                    loss = (loss_cls + loss_mag + loss_ev) * g_store

                    (loss / max(1, grad_accum_steps)).backward()
                    step_mod += 1
                    if step_mod % max(1, grad_accum_steps) == 0:
                        if max_grad_norm is not None:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                        opt.step(); opt.zero_grad(set_to_none=True)

                    bs = yb.size(0)
                    sum_loss += loss.item() * bs
                    n_obs += bs

                # 남은 그라드 플러시
                if step_mod % max(1, grad_accum_steps) != 0:
                    if max_grad_norm is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    opt.step(); opt.zero_grad(set_to_none=True)

                train_losses.append(sum_loss / max(1, n_obs))

                # ===== Validation =====
                if use_validation and len(Xval) > 0:
                    model.eval()
                    with torch.no_grad():
                        y_hat_v, prob_logit_v, resid_raw_v, base_mix_v = model(
                            Xval, wdval, ssval, mmval,
                            future_weekday=fwd_val, future_month=fmm_val, future_prox=fpx_val,
                            base_last=make_mixed_base(Xval, PREDICT),
                            return_components=True
                        )
                        yb_v   = yval
                        ybin_v = (yb_v > 0).float()
                        mask_pos_v = (ybin_v > 0.5)

                        r_scale_v = F.softplus(model.r_scale_param)
                        pred_mag_v = torch.relu(base_mix_v + r_scale_v * torch.tanh(resid_raw_v))

                        # BCE(약화)
                        pos_rate_v = ybin_v.mean().clamp(1e-3, 1 - 1e-3)
                        pos_weight_v = ((1 - pos_rate_v) / pos_rate_v).clamp(1.0, 4.0)
                        loss_cls_v = F.binary_cross_entropy_with_logits(
                            prob_logit_v, ybin_v, pos_weight=pos_weight_v
                        ).mean() * lambda_cls

                        # Magnitude(양성일)
                        if mask_pos_v.any():
                            loss_mag_v = F.smooth_l1_loss(pred_mag_v[mask_pos_v], yb_v[mask_pos_v]) * lambda_mag
                        else:
                            loss_mag_v = torch.tensor(0.0, device=yb_v.device)

                        # real sMAPE (0-day 제외)
                        yhat_real_v = pred_mag_v * scale + y_min
                        yb_real_v   = yb_v       * scale + y_min
                        loss_ev_v   = smape_real(yhat_real_v, yb_real_v, eps=1.0, ignore_zero_target=True) * lambda_ev

                        v = (loss_cls_v + loss_mag_v + loss_ev_v).item() * g_store
                        val_losses.append(v); sch.step(v)

                        # 로깅용 real sMAPE (그냥 지표 느낌값)
                        v_smape = smape_real(yhat_real_v, yb_real_v, eps=1.0, ignore_zero_target=True).item()
                        val_smape_list.append(v_smape)

                        if early.step(v, model, ep):
                            print(f"[{menu_key}] Early stop @ {ep+1} | best val={min(val_losses):.6f} | best_realSMAPE={min(val_smape_list):.4f}")
                            break

            # 마지막 LOOKBACK 저장
            last_seq = {
                'X_num': feat[FEATURES].values[-LOOKBACK:],
                'weekday': feat['weekday'].values[-LOOKBACK:],
                'season':  feat['season'].values[-LOOKBACK:],
                'month_idx': feat['month_idx'].values[-LOOKBACK:]
            }
            lower_bound = 1.0

            trained[menu_key] = {
                'model': model.eval(),
                'scalers': (scaler_y, scaler_rm, scaler_delta),
                'last_sequence': last_seq,
                'lower_bound': float(lower_bound),
                'feature_order': FEATURES,
            }

            visualize_loss(train_losses, val_losses if use_validation else None,
                           f"{menu_key}_ITEMWISE", save=True, out_dir=plot_dir, val_smape=val_smape_list)

            pbar.set_postfix_str(_menu_only_from_key(menu_key)[:28]); pbar.update(1)

    return trained


def predict_nhits_itemwise(test_df: pd.DataFrame,
                           trained: dict,
                           test_prefix: str,
                           *,
                           discontinued: dict[str, str | pd.Timestamp] | None = None,
                           rule: str = 'after',
                           grace_days: int = 0,
                           allow_zero: bool = False) -> pd.DataFrame:
    """
    메뉴 단일 모델로 예측 수행.
    trained 는 train_nhits_itemwise가 반환한 dict[영업장명_메뉴명] 이어야 함.
    """
    results = []

    # 단종 처리 준비
    cutoff_map = None
    if discontinued is not None:
        def _to_ts(v): return v if isinstance(v, pd.Timestamp) else pd.to_datetime(v)
        cutoff_map = {k: _to_ts(v) for k, v in discontinued.items()}
        if grace_days != 0:
            for k in cutoff_map: cutoff_map[k] = cutoff_map[k] + pd.Timedelta(days=grace_days)

    # 휴일 인덱스
    global HOLIDAYS_INT
    if HOLIDAYS_INT is None:
        yrs = pd.to_datetime(test_df['영업일자']).dt.year
        y_min, y_max = int(yrs.min()), int(yrs.max())
        HOLIDAYS_INT = build_holidays_int(range(y_min - 1, y_max + 2))

    df = test_df.copy()
    if '영업장명' not in df.columns:
        df['영업장명'] = df['영업장명_메뉴명'].apply(_store_from_key)
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    for menu_key, st_menu in df.groupby('영업장명_메뉴명', sort=False):
        if menu_key not in trained:
            continue

        pack = trained[menu_key]
        model = pack['model']
        scaler_y, scaler_rm, scaler_delta = pack['scalers']
        last_seq = pack['last_sequence']
        lower_bound = pack['lower_bound']

        st = _store_from_key(menu_key)


        st_menu = st_menu.sort_values('영업일자').copy()
        ft = build_features(st_menu, scaler_y, scaler_rm, scaler_delta, fit=False, date_col='영업일자')

        
        # 입력 시퀀스 확보
        if len(ft) < LOOKBACK:
            if last_seq is None:
                continue
            x_num_np   = np.asarray(last_seq['X_num'], dtype=np.float32)
            weekday_np = np.asarray(last_seq['weekday'], dtype=np.int64)
            season_np  = np.asarray(last_seq['season'],  dtype=np.int64)
            month_np   = np.asarray(last_seq['month_idx'], dtype=np.int64)
            last_obs_date = pd.to_datetime(st_menu['영업일자'].max())
        else:
            recent = ft.iloc[-LOOKBACK:].copy()
            x_num_np   = recent[FEATURES].values.astype(np.float32)
            weekday_np = recent['weekday'].values.astype(np.int64)
            season_np  = recent['season'].values.astype(np.int64)
            month_np   = recent['month_idx'].values.astype(np.int64)
            last_obs_date = pd.to_datetime(st_menu['영업일자'].max())

        # 텐서
        x_num_input = torch.tensor(x_num_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        weekday_seq = torch.tensor(weekday_np, dtype=torch.long,   device=DEVICE).unsqueeze(0)
        season_seq  = torch.tensor(season_np,  dtype=torch.long,   device=DEVICE).unsqueeze(0)
        month_seq   = torch.tensor(month_np,   dtype=torch.long,   device=DEVICE).unsqueeze(0)

        # 미래 달력 (진짜 날짜)
        horizon_dates = pd.date_range(start=last_obs_date + pd.Timedelta(days=1),
                                      periods=PREDICT, freq='D')
        future_wd = torch.tensor([d.dayofweek for d in horizon_dates], device=DEVICE, dtype=torch.long).unsqueeze(0)
        future_mm = torch.tensor([d.month-1 for d in horizon_dates], device=DEVICE, dtype=torch.long).unsqueeze(0)
        future_px_np = holiday_prox_searchsorted(horizon_dates, HOLIDAYS_INT, K=10)
        future_px = torch.tensor(future_px_np, device=DEVICE).unsqueeze(0)

        # 추론
        model.eval()
        with torch.no_grad():
            base_last = make_mixed_base(x_num_input, PREDICT)  # (B,H)

            y_hat, prob_logit, resid_raw, base_mix = model(
                x_num_input, weekday_seq, season_seq, month_seq,
                future_weekday=future_wd, future_month=future_mm, future_prox=future_px,
                base_last=base_last, return_components=True
            )
            # tuple 언팩
            pred_scaled = y_hat.squeeze(0).detach().cpu().numpy()

        vals_real = inverse_clipped_from_scaler(scaler_y, pred_scaled)

        # 하한값: 리더보드 0-day 제외면 1.0 유지; allow_zero=True면 0 허용
        if not allow_zero and lower_bound > 0.0:
            vals_real = np.maximum(vals_real, lower_bound)

        # 단종 처리
        if cutoff_map is not None and menu_key in cutoff_map:
            cutoff = cutoff_map[menu_key]
            zero_mask = (horizon_dates >= cutoff) if rule == 'on_or_after' else (horizon_dates > cutoff)
            vals_real = np.where(zero_mask, 0.0, vals_real)

        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        for d, v in zip(pred_dates, vals_real):
            results.append({'영업일자': d, '영업장명_메뉴명': menu_key, '매출수량': float(v)})

    return pd.DataFrame(results, columns=['영업일자','영업장명_메뉴명','매출수량'])
# -------------------------
# 제출 포맷 변환
# -------------------------
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_dict = dict(zip(zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']), pred_df['매출수량']))
    final_df = sample_submission.copy()
    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
    return final_df


In [185]:
#Data load
train = pd.read_csv('./train/train.csv')
train = generate_combined_holiday_list(train, solar_md_holidays, lunar_solar_dates)
train = filter_all_menus_by_leading_zeros(
    train,
    min_zero_days=90,
    apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
)
trained_models = train_nhits_itemwise(train, use_validation=True)

Training (item-wise):   0%|          | 0/193 [00:00<?, ?it/s]

[느티나무 셀프BBQ_1인 수저세트] Early stop @ 24 | best val=0.308480 | best_realSMAPE=0.5424


Training (item-wise):   1%|          | 2/193 [00:08<11:46,  3.70s/it, BBQ55(단체)] 

[느티나무 셀프BBQ_BBQ55(단체)] Early stop @ 11 | best val=0.275106 | best_realSMAPE=0.4399


Training (item-wise):   2%|▏         | 3/193 [00:13<13:57,  4.41s/it, 대여료 30,000원]

...

[화담숲카페_메밀미숫가루] Early stop @ 14 | best val=0.265119 | best_realSMAPE=0.4401


Training (item-wise):  99%|█████████▉| 192/193 [11:28<00:04,  4.03s/it, 카페라떼 ICE]  

[화담숲카페_카페라떼 ICE] Early stop @ 13 | best val=0.252779 | best_realSMAPE=0.3980


Training (item-wise): 100%|██████████| 193/193 [11:31<00:00,  3.58s/it, 현미뻥스크림]

[화담숲카페_현미뻥스크림] Early stop @ 24 | best val=0.314535 | best_realSMAPE=0.5426


In [186]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))
for path in test_files:
    test_df = pd.read_csv(path)
    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_nhits_itemwise(
        test_df, trained_models, test_prefix,
        discontinued=DISCONTINUED,
        rule='after',      # 단종일 "이후" 0
        grace_days=0       # 유예일 없으면 0
    )
    all_preds.append(pred_df)
    #display(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

In [187]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('./Prediction/model_v9_25.csv', index=False, encoding='utf-8-sig')
result = pd.read_csv('./Prediction/model_v9_25.csv')
display(result.head())

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_66506/3921510437.py:1185: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '10.091506004333496' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_66506/3921510437.py:1185: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.198326587677002' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_66506/3921510437.py:1185: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.454607009887695' has 

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,10.091506,2.198327,6.454607,2.764315,1.712311,2.220259,1.000000,3.069888,4.437922,...,7.013157,21.591093,26.372896,6.350276,56.985027,33.991962,6.072524,28.701174,10.846137,25.381863
1,TEST_00+2일,4.368672,23.273048,4.253702,2.279413,1.320227,1.590381,2.255558,2.836798,2.903657,...,10.277397,11.301176,20.093922,12.248416,53.670120,21.722395,8.999277,28.785803,12.925571,25.481930
2,TEST_00+3일,6.053117,20.286369,4.086580,3.120935,1.000000,2.143508,1.000000,5.767039,4.056156,...,4.328284,8.511633,11.056345,6.441166,34.520233,37.588596,1.000000,11.115778,4.091214,6.797244
3,TEST_00+4일,7.895769,30.708778,2.999492,3.541858,1.160798,1.037759,1.191134,2.545672,4.000645,...,4.223518,9.134962,8.409593,5.177320,28.580357,20.475821,6.751208,15.390725,6.585247,3.560796
4,TEST_00+5일,7.855635,32.896000,3.569655,2.885067,1.432849,2.343064,4.898468,10.173889,2.917064,...,4.198689,10.407540,9.809881,3.242750,33.576370,42.283211,3.521693,43.205624,12.165896,21.582926
